# Day 7

*Transforms the VIIRS dataset into a orthographic projection*

## 1. Configuration

In [19]:
from osgeo import gdal
gdal.UseExceptions()

In [20]:
import glob
import numpy as np
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt

In [21]:
PREFIX = "VIIRS_Composite_Global_Oct2024"

# Top row of tiles only (row offset 0000000000); skips the Antarctic strips
TILE_GLOB = f"{PREFIX}-0000000000-*.tif"

# Orthographic view: tweak LAT_0 / LON_0 to tilt the globe
LAT_0, LON_0 = 5, -80
PROJ = f"+proj=ortho +lat_0={LAT_0} +lon_0={LON_0} +R=6371000 +units=m +no_defs"

R = 6_371_000
EXTENT = (-R, -R, R, 6_000_000)

RES = 500  # output pixel size in metres (300 = more detail, bigger file)

In [22]:
VRT = "top.vrt"
OUT_LIGHTS = "ortho_lights.tif"
OUT_AURORA = "ortho_aurora.tif"
OUT_FINAL = "ortho_combined.tif"
PREVIEW_SCALE = 4

## 2. Helper Functions

In [24]:
def progress(fraction, message, data):
    print(f"\rprogress: {fraction*100:5.1f}%", end="", flush=True)
    return 1  # continue

def build_vrt(vrt_path, tiles):
    ds = gdal.BuildVRT(vrt_path, tiles)
    ds = None  # close so the .vrt is written to disk

def warp(src, dst):
    """Reproject src onto the orthographic grid, streaming from disk."""
    ds = gdal.Warp(
        dst, src,
        dstSRS=PROJ,
        outputBounds=EXTENT,
        xRes=RES, yRes=RES,
        resampleAlg="average",          # avoids aliasing at high latitudes
        multithread=True,
        warpMemoryLimit=2048,           # MB
        warpOptions=["NUM_THREADS=ALL_CPUS"],
        creationOptions=["TILED=YES", "BLOCKXSIZE=512", "BLOCKYSIZE=512",
                         "COMPRESS=DEFLATE", "BIGTIFF=YES"],
        callback=progress,
    )
    ds = None  # flush and close
    print(f"\nWrote {dst}")

def composite_over(top_path, bottom_path, out_path, chunk=2048):
    """Alpha-composite top over bottom, window by window (low memory)."""
    with rasterio.open(bottom_path) as bot, rasterio.open(top_path) as top:
        assert bot.shape == top.shape, "Layers must share the same grid"
        profile = bot.profile.copy()
        profile.update(count=4, dtype="uint8", tiled=True,
                       blockxsize=512, blockysize=512,
                       compress="deflate", BIGTIFF="YES")
        with rasterio.open(out_path, "w", **profile) as dst:
            for row in range(0, bot.height, chunk):
                for col in range(0, bot.width, chunk):
                    w = Window(col, row,
                               min(chunk, bot.width - col),
                               min(chunk, bot.height - row))
                    b = bot.read(window=w).astype("float32") / 255.0
                    t = top.read(window=w).astype("float32") / 255.0
                    ta, ba = t[3], b[3]
                    oa = ta + ba * (1 - ta)
                    safe = np.where(oa == 0, 1, oa)
                    out = np.empty_like(b)
                    for i in range(3):
                        out[i] = (t[i] * ta + b[i] * ba * (1 - ta)) / safe
                    out[3] = oa
                    dst.write((out * 255).clip(0, 255).astype("uint8"), window=w)
    print(f"Composite written: {out_path}")

def preview(path, scale=PREVIEW_SCALE):
    """Decimated read so matplotlib never sees the full-res array."""
    with rasterio.open(path) as src:
        img = src.read(
            out_shape=(src.count, src.height // scale, src.width // scale),
            resampling=rasterio.enums.Resampling.average,
        )
    img = img.transpose(1, 2, 0)
    fig, ax = plt.subplots(figsize=(12, 12), facecolor="black")
    ax.imshow(img)
    ax.axis("off")
    plt.show()

## 3. Build mosaic and warp

In [31]:
tiles = sorted(glob.glob(TILE_GLOB))
assert tiles, f"No tiles matched {TILE_GLOB}"
print(f"Using {len(tiles)} tiles")

build_vrt(VRT, tiles)
warp(VRT, OUT_LIGHTS)
final = OUT_LIGHTS

Using 3 tiles
progress: 100.0%
Wrote ortho_lights.tif


## 4. Download image

In [34]:
import os
gdal.SetConfigOption("GDAL_PAM_ENABLED", "NO")  # no .aux.xml sidecar file

# Use the combined file if you made one, otherwise the lights-only warp
final = OUT_FINAL if (AURORA_FILE and os.path.exists(OUT_FINAL)) else OUT_LIGHTS
print(f"Source: {final} ({os.path.getsize(final)/1e9:.2f} GB)")

ds = gdal.Translate(
    "ortho_lights.png", final,
    format="PNG",
    creationOptions=["ZLEVEL=6"],
    callback=progress,
)
ds = None
print("\nSaved ortho_lights.png")

Source: ortho_lights.tif (0.46 GB)
progress: 100.0%
Saved ortho_lights.png
